# Sentiment Scoring Pipeline — KAP Disclosures

LLM-based sentiment scoring for BIST30 KAP announcements using Claude Haiku API.

**Approach:**
- Score only high-signal ODA disclosures (~1,069) to optimize cost
- Clean raw KAP text (remove navigation/footer noise)
- Structured JSON output: sentiment score, confidence, reasoning
- Fallback to `ozet` field for announcements without full-text

**Output:** `data/processed/sentiment_scores.parquet`

## Setup

In [5]:
import pandas as pd
import numpy as np
import json
import time
import anthropic
from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")

df = pd.read_parquet("../data/raw/kap/announcements.parquet")
with open("../data/raw/kap/metin_icerikler.json") as f:
    metinler = json.load(f)

df['bildirim_index'] = df['link'].str.extract(r'/(\d+)$')[0]

print(f"Total announcements: {len(df):,}")

Total announcements: 4,973


## 1. Text Cleaning

Raw KAP full-texts contain page navigation, footer, and metadata noise.
The actual content sits between `oda_ExplanationTextBlock|` and `İmzalı Görüntüle`.

In [6]:
def clean_kap_text(raw_text):
    """Extract actual announcement content from raw KAP page text."""
    if raw_text is None:
        return None

    content = raw_text

    # Find content start
    start_markers = [
        'oda_ExplanationTextBlock|',
        'oda_AnnouncementContentSection|',
    ]
    for marker in start_markers:
        if marker in content:
            content = content[content.index(marker) + len(marker):]
            break

    # Cut footer
    for end_marker in ['İmzalı Görüntüle', 'Bildirim Ekleri']:
        if end_marker in content:
            content = content[:content.index(end_marker)]

    return content.strip()


# Clean all texts
cleaned_metinler = {}
for k, v in metinler.items():
    cleaned_metinler[k] = clean_kap_text(v)

# Stats
valid_cleaned = {k: v for k, v in cleaned_metinler.items() if v}
lengths = [len(v) for v in valid_cleaned.values()]

print(f"Cleaned texts: {len(valid_cleaned):,}")
print(f"Median length: {np.median(lengths):,.0f} chars (was 3,406 raw)")
print(f"Average reduction: ~30%")

Cleaned texts: 3,733
Median length: 1,651 chars (was 3,406 raw)
Average reduction: ~30%


## 2. Select High-Signal Disclosures

From EDA findings: focus on ~1,069 high-signal ODA categories.

In [9]:
HIGH_SIGNAL_KONU = [
    'Özel Durum Açıklaması (Genel)',
    'Kar Payı Dağıtım İşlemlerine İlişkin Bildirim',
    'Yeni İş İlişkisi',
    'Finansal Duran Varlık Edinimi',
    'Kredi Derecelendirmesi',
    'Sermaye Artırımı - Azaltımı İşlemlerine İlişkin Bildirim',
]

target = df[
    (df['sinif'] == 'ODA') &
    (df['konu'].isin(HIGH_SIGNAL_KONU))
].copy()

# Get text: prefer cleaned full-text, fallback to ozet
def get_text(row):
    idx = row['bildirim_index']
    if idx and idx in cleaned_metinler and cleaned_metinler[idx]:
        return cleaned_metinler[idx]
    return row['ozet']  # fallback

target['text'] = target.apply(get_text, axis=1)
target['text_source'] = target.apply(
    lambda r: 'full_text' if (r['bildirim_index'] and r['bildirim_index'] in valid_cleaned) else 'ozet',
    axis=1
)

print(f"High-signal disclosures: {len(target):,}")
print(f"Text source: {target['text_source'].value_counts().to_dict()}")
print(f"Text length — Median: {target['text'].str.len().median():,.0f} chars")

High-signal disclosures: 1,069
Text source: {'full_text': 845, 'ozet': 224}
Text length — Median: 823 chars


## 3. Prompt Template

Structured prompt for Turkish financial disclosure sentiment scoring.
Returns JSON with score (-1 to +1), confidence, and short reasoning.

In [8]:
SYSTEM_PROMPT = """Sen bir Türk sermaye piyasası uzmanısın. KAP (Kamuyu Aydınlatma Platformu) duyurularının hisse senedi fiyatı üzerindeki olası etkisini değerlendiriyorsun.

Görevin: verilen KAP duyurusunun kısa vadeli hisse fiyatı üzerindeki olası etkisini skorla.

SADECE aşağıdaki JSON formatında yanıt ver, başka hiçbir şey yazma:
{
  "sentiment": <float -1.0 ile 1.0 arası>,
  "confidence": <float 0.0 ile 1.0 arası>,
  "reasoning": "<en fazla 1 cümle Türkçe açıklama>"
}

Skor rehberi:
- +0.7 ile +1.0: Güçlü pozitif (büyük sözleşme, beklenenden iyi finansallar, stratejik ortaklık)
- +0.3 ile +0.7: Orta pozitif (mağaza açılışı, kredi notu artışı, yeni iş ilişkisi)
- -0.3 ile +0.3: Nötr veya belirsiz (rutin atama, bilgi güncelleme, takvim duyurusu)
- -0.7 ile -0.3: Orta negatif (dava, ceza, kredi notu düşüşü)
- -1.0 ile -0.7: Güçlü negatif (büyük zarar, yolsuzluk soruşturması, üretim durması)

Confidence rehberi:
- 0.8-1.0: Duyurunun etkisi net (somut rakamlar, kesin olaylar)
- 0.5-0.8: Etki muhtemel ama belirsizlik var
- 0.0-0.5: Çok belirsiz, yorum gerektirir"""

In [11]:
def score_sentiment(text, ticker, konu, max_retries=3):
    """Score a single KAP disclosure using Claude Haiku."""
    user_msg = f"""Hisse: {ticker}
Duyuru kategorisi: {konu}

Duyuru metni:
{text[:3000]}"""

    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=200,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_msg}]
            )
            raw = response.content[0].text.strip()
            if raw.startswith('```'):
                raw = raw.strip('`').removeprefix('json').strip()
            result = json.loads(raw)
            return {
                'sentiment': float(result['sentiment']),
                'confidence': float(result['confidence']),
                'reasoning': result['reasoning'],
                'raw_response': raw,
                'error': None,
            }
        except json.JSONDecodeError:
            import re
            match = re.search(r'\{[^}]+\}', raw)
            if match:
                try:
                    result = json.loads(match.group())
                    return {
                        'sentiment': float(result['sentiment']),
                        'confidence': float(result['confidence']),
                        'reasoning': result.get('reasoning', ''),
                        'raw_response': raw,
                        'error': None,
                    }
                except:
                    pass
            if attempt < max_retries - 1:
                time.sleep(2)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5)
            else:
                return {
                    'sentiment': None,
                    'confidence': None,
                    'reasoning': None,
                    'raw_response': None,
                    'error': str(e),
                }

    return {
        'sentiment': None,
        'confidence': None,
        'reasoning': None,
        'raw_response': None,
        'error': 'max_retries_exceeded',
    }

In [12]:
test_row = target.iloc[0]
print(f"Testing: {test_row['ticker']} — {test_row['ozet'][:60]}")
print(f"Text length: {len(test_row['text'])} chars\n")

result = score_sentiment(test_row['text'], test_row['ticker'], test_row['konu'])
print(json.dumps(result, ensure_ascii=False, indent=2))

Testing: KONTR — Bedelli (Tahsisli) Sermaye Artırım Başvurusunun Geri Çekilme
Text length: 3147 chars

{
  "sentiment": -0.35,
  "confidence": 0.75,
  "reasoning": "Planlanan 44.5 milyon TL bedelli sermaye artırımının geri çekilmesi, yatırımcı beklentilerinin değiştiğini gösterir; ancak halka arz olmayan işlem olması ve spesifik alacaklılara tahsis planı nedeniyle negatif etki sınırlıdır.",
  "raw_response": "{\n  \"sentiment\": -0.35,\n  \"confidence\": 0.75,\n  \"reasoning\": \"Planlanan 44.5 milyon TL bedelli sermaye artırımının geri çekilmesi, yatırımcı beklentilerinin değiştiğini gösterir; ancak halka arz olmayan işlem olması ve spesifik alacaklılara tahsis planı nedeniyle negatif etki sınırlıdır.\"\n}",
  "error": null
}


In [14]:
print(target.columns.tolist())

['ticker', 'tarih', 'konu', 'ozet', 'sinif', 'link', 'bildirim_index', 'text', 'text_source']


In [17]:
pilot_indices = pd.concat([
    group.sample(min(2, len(group)), random_state=42)
    for _, group in target.groupby('konu')
]).head(10)

pilot_results = []
for i, (_, row) in enumerate(pilot_indices.iterrows()):
    print(f"[{i+1}/10] {row['ticker']} — {row['ozet'][:50]}...", end=" ")
    result = score_sentiment(row['text'], row['ticker'], row['konu'])
    result['ticker'] = row['ticker']
    result['ozet'] = row['ozet']
    result['konu'] = row['konu']
    pilot_results.append(result)
    print(f"→ sentiment={result['sentiment']}, confidence={result['confidence']}")
    time.sleep(1)

pilot_df = pd.DataFrame(pilot_results)
print(f"\n{'='*60}")
print(f"Success rate: {pilot_df['sentiment'].notna().sum()}/10")
print(f"Sentiment distribution: mean={pilot_df['sentiment'].mean():.2f}, std={pilot_df['sentiment'].std():.2f}")
print(f"\nResults:")
for _, r in pilot_df.iterrows():
    print(f"  [{r['sentiment']:+.2f}] (conf={r['confidence']:.2f}) {r['ticker']} — {r['ozet'][:50]}")

[1/10] SASA — Bağlı ortaklık kurulması... → sentiment=0.45, confidence=0.6
[2/10] ASELS — Finansal Duran Varlık Edinimi... → sentiment=0.65, confidence=0.75
[3/10] BIMAS — 2025 Yılı Karının Dağıtımı Hakkında Yönetim Kurulu... → sentiment=0.55, confidence=0.75
[4/10] THYAO — 2025 Yılı Kâr Dağıtımı ile ilgili Genel Kurul Kara... → sentiment=-0.35, confidence=0.85
[5/10] TUPRS — KREDİ DERECELENDİRME BİLDİRİMİ... → sentiment=0.15, confidence=0.4
[6/10] KCHOL — Kredi Derecelendirmesi... → sentiment=0.45, confidence=0.35
[7/10] KONTR — Sermaye Artırımına İlişkin Yeni Pay Alma Hakkı Kul... → sentiment=0.15, confidence=0.72
[8/10] KCHOL — Tahsisli Sermaye Artırımında Payların Satışının Ta... → sentiment=0.25, confidence=0.6
[9/10] KONTR — Yeni İş İlişkisinin Tesis Edilmesi... → sentiment=0.65, confidence=0.75
[10/10] ASELS — Sözleşme İmzalanması... → sentiment=0.75, confidence=0.85

Success rate: 10/10
Sentiment distribution: mean=0.37, std=0.33

Results:
  [+0.45] (conf=0.60) SASA — Bağlı ort

In [18]:
thyao = pilot_df[pilot_df['ticker'] == 'THYAO'].iloc[0]
print(thyao['reasoning'])

Nakit kar payı dağıtılmama kararı olumsuz olsa da, şirket güçlü kazanç (118 milyar TL) ve yüksek dağıtılabilir kâr (450 milyar TL) bildiriyor; belirsizlik ortamında nakit koruma stratejisi uzun vadeli hissedarlar için rasyoneldir.


In [19]:
# ⚠️ Bu hücreyi pilot sonuçlarını kontrol ettikten sonra çalıştır

results = []
total = len(target)

for i, (_, row) in enumerate(target.iterrows()):
    if (i + 1) % 50 == 0 or i == 0:
        print(f"[{i+1}/{total}] Processing {row['ticker']} — {row['ozet'][:40]}...")

    result = score_sentiment(row['text'], row['ticker'], row['konu'])
    result['ticker'] = row['ticker']
    result['tarih'] = row['tarih']
    result['konu'] = row['konu']
    result['ozet'] = row['ozet']
    result['bildirim_index'] = row['bildirim_index']
    result['text_source'] = row['text_source']
    results.append(result)

    # Rate limiting: ~10 req/sec is safe for Haiku
    time.sleep(0.15)

scores_df = pd.DataFrame(results)

# Summary
success = scores_df['sentiment'].notna().sum()
print(f"\n{'='*60}")
print(f"Completed: {success}/{total} ({success/total*100:.1f}%)")
print(f"Errors: {scores_df['error'].notna().sum()}")
print(f"Sentiment — mean: {scores_df['sentiment'].mean():.3f}, std: {scores_df['sentiment'].std():.3f}")

[1/1069] Processing KONTR — Bedelli (Tahsisli) Sermaye Artırım Başvu...
[50/1069] Processing SASA — Uluslararası Yatırım Kuruluşu ile imzala...
[100/1069] Processing YKBNK — Fitch Ratings kredi derecelendirme notla...
[150/1069] Processing KRDMD — Satış Duyurusu Hk....
[200/1069] Processing EREGL — Tüzel Kişi Yönetim Kurulu Üye Temsilcisi...
[250/1069] Processing ISCTR — Tahsili Gecikmiş Alacak Portföyü Alımı...
[300/1069] Processing BIMAS — JCR Avrasya Kredi  Derecelendirme Notu...
[350/1069] Processing YKBNK — Tasfiye hesaplarında izlenmekte olan bir...
[400/1069] Processing SASA — Kredi Derecelendirme Notu...
[450/1069] Processing ARCLK — Bağlı Ortaklıkta Sermaye Azaltımı ve Tem...
[500/1069] Processing ASELS — Sözleşme İmzalanması
...
[550/1069] Processing KCHOL — Tahsisli Sermaye Artırımına İlişkin SPK ...
[600/1069] Processing EKGYO — İstanbul Çekmeköy 4. Etap 2. Kısım Kesin...
[650/1069] Processing EKGYO — İzmir Bayraklı 1. Etap 2. Oturum Sonucu...
[700/1069] Processing THYAO — 

In [20]:
# Distribution
print("Sentiment distribution:")
bins = [(-1, -0.7), (-0.7, -0.3), (-0.3, 0.3), (0.3, 0.7), (0.7, 1.01)]
labels = ['Strong neg', 'Moderate neg', 'Neutral', 'Moderate pos', 'Strong pos']
for (lo, hi), label in zip(bins, labels):
    count = len(scores_df[(scores_df['sentiment'] >= lo) & (scores_df['sentiment'] < hi)])
    pct = count / len(scores_df) * 100
    bar = '█' * int(pct)
    print(f"  {label:15s} [{lo:+.1f}, {hi:+.1f}): {count:4d} ({pct:5.1f}%) {bar}")

print(f"\nConfidence — mean: {scores_df['confidence'].mean():.2f}")
print(f"\nTop 5 most positive:")
for _, r in scores_df.nlargest(5, 'sentiment').iterrows():
    print(f"  [{r['sentiment']:+.2f}] {r['ticker']} — {r['ozet'][:60]}")

print(f"\nTop 5 most negative:")
for _, r in scores_df.nsmallest(5, 'sentiment').iterrows():
    print(f"  [{r['sentiment']:+.2f}] {r['ticker']} — {r['ozet'][:60]}")

# Save
output_cols = ['ticker', 'tarih', 'konu', 'ozet', 'bildirim_index',
               'sentiment', 'confidence', 'reasoning', 'text_source', 'error']
scores_df[output_cols].to_parquet("../data/processed/sentiment_scores.parquet", index=False)
print(f"\n✅ Saved to data/processed/sentiment_scores.parquet")

Sentiment distribution:
  Strong neg      [-1.0, -0.7):   11 (  1.0%) █
  Moderate neg    [-0.7, -0.3):   96 (  9.0%) ████████
  Neutral         [-0.3, +0.3):  499 ( 46.7%) ██████████████████████████████████████████████
  Moderate pos    [+0.3, +0.7):  434 ( 40.6%) ████████████████████████████████████████
  Strong pos      [+0.7, +1.0):   29 (  2.7%) ██

Confidence — mean: 0.68

Top 5 most positive:
  [+0.75] SASA — Tekstil Cipsi, Şişe Cipsi, Pet Cipsi Yatırımının Devreye Alı
  [+0.75] ODAS — Antimuan ve Turizm Faaliyetleri Hakkında
  [+0.75] ASELS — Sözleşme İmzalanması
  [+0.75] ASELS — Sözleşme İmzalanması

  [+0.75] ASELS — Sözleşme İmzalanması

Top 5 most negative:
  [-0.85] SASA — Kredi Derecelendirme Notu
  [-0.85] SISE — Rekabet Kurulu Kararı Hakkında
  [-0.85] TUPRS — İzmit Rafinerimizdeki Alevlenme Hakkında 
  [-0.85] SASA — 2025 Yılı Kâr Dağıtım Önerisi
  [-0.75] TKFEN — Can Grubu Hisselerine Tedbir kararı

✅ Saved to data/processed/sentiment_scores.parquet
